# 02 — Data Quality & Validation

## Retention Intelligence Platform

### Objective

Determine whether the acquired IBM Telco Customer Churn dataset is structurally and statistically suitable for downstream analysis and machine-learning development.

This notebook performs:

1. Provenance verification
2. Schema validation
3. Data integrity checks
4. Missing-value analysis
5. Duplicate analysis
6. Identifier validation
7. Data-type validation
8. Numerical range validation
9. Categorical-domain validation
10. Target validation
11. Data profiling
12. Quality-gate decision

### Important rule

This notebook does not perform feature engineering or modeling.

EDA begins only after this quality stage passes.

## Load configuration

In [44]:
from pathlib import Path
import pandas as pd
import numpy as np
import json

PROJECT_ROOT = Path('../retention-intelligence')

RAW_FILE = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "telco"
    / "Telco-Customer-Churn.csv"
)
REPORT_DIR = (
    PROJECT_ROOT
    / "data"
    / "reports"
    / "telco"
)

REPORT_DIR.mkdir(parents=True, exist_ok=True)

## Load the raw dataset

In [14]:
df = pd.read_csv(RAW_FILE)

print(f"Rows:    {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")

Rows:    7,043
Columns: 21


## Schema Inspection

In [15]:
schema = pd.DataFrame({
    "column": df.columns,
    "dtype": df.dtypes.astype(str).values,
    "non_null": df.notna().sum().values,
    "null_count": df.isna().sum().values,
    "unique_values": df.nunique(dropna=False).values,
})

schema

,column,dtype,non_null,null_count,unique_values
0,customerID,str,7043,0,7043
1,gender,str,7043,0,2
2,SeniorCitizen,int64,7043,0,2
3,Partner,str,7043,0,2
4,Dependents,str,7043,0,2
5,tenure,int64,7043,0,73
6,PhoneService,str,7043,0,2
7,MultipleLines,str,7043,0,3
8,InternetService,str,7043,0,3
9,OnlineSecurity,str,7043,0,3


### Required columns

In [17]:
schema = pd.DataFrame({
    "column": df.columns,
    "dtype": df.dtypes.astype(str).values,
    "non_null": df.notna().sum().values,
    "null_count": df.isna().sum().values,
    "unique_values": df.nunique(dropna=False).values,
})

schema

,column,dtype,non_null,null_count,unique_values
0,customerID,str,7043,0,7043
1,gender,str,7043,0,2
2,SeniorCitizen,int64,7043,0,2
3,Partner,str,7043,0,2
4,Dependents,str,7043,0,2
5,tenure,int64,7043,0,73
6,PhoneService,str,7043,0,2
7,MultipleLines,str,7043,0,3
8,InternetService,str,7043,0,3
9,OnlineSecurity,str,7043,0,3


In [18]:
EXPECTED_COLUMNS = {
    "customerID",
    "gender",
    "SeniorCitizen",
    "Partner",
    "Dependents",
    "tenure",
    "PhoneService",
    "MultipleLines",
    "InternetService",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
    "Contract",
    "PaperlessBilling",
    "PaymentMethod",
    "MonthlyCharges",
    "TotalCharges",
    "Churn",
}

In [19]:
actual_columns = set(df.columns)

missing_columns = EXPECTED_COLUMNS - actual_columns
unexpected_columns = actual_columns - EXPECTED_COLUMNS

print("Missing columns:", missing_columns)
print("Unexpected columns:", unexpected_columns)

Missing columns: set()
Unexpected columns: set()


In [20]:
assert not missing_columns, f"Missing required columns: {missing_columns}"

## Duplicate validation

In [21]:
duplicate_rows = df.duplicated().sum()
duplicate_customer_ids = df["customerID"].duplicated().sum()

print(f"Duplicate rows: {duplicate_rows}")
print(f"Duplicate customer IDs: {duplicate_customer_ids}")

Duplicate rows: 0
Duplicate customer IDs: 0


In [22]:
assert duplicate_customer_ids == 0, (
    "Duplicate customer IDs detected."
)

## Missing-value analysis

In [23]:
missing = (
    df.isna().sum().to_frame("missing_count")
)

missing["missing_pct"] = (
    missing["missing_count"] / len(df) * 100
)

missing.sort_values("missing_pct", ascending=False)

,missing_count,missing_pct
customerID,0,0.0
DeviceProtection,0,0.0
TotalCharges,0,0.0
MonthlyCharges,0,0.0
PaymentMethod,0,0.0
PaperlessBilling,0,0.0
Contract,0,0.0
StreamingMovies,0,0.0
StreamingTV,0,0.0
TechSupport,0,0.0


## Data type validation

In [24]:
df.dtypes

customerID              str
gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges            str
Churn                   str
dtype: object

In [25]:
numeric_columns = [
    "SeniorCitizen",
    "tenure",
    "MonthlyCharges",
]

for column in numeric_columns:
    print(column, pd.api.types.is_numeric_dtype(df[column]))

SeniorCitizen True
tenure True
MonthlyCharges True


In [26]:
pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
).isna().sum()

np.int64(11)

## Numerical range validation

In [27]:
range_checks = {
    "SeniorCitizen": (0, 1),
    "tenure": (0, None),
    "MonthlyCharges": (0, None),
}

In [28]:
for column, (minimum, maximum) in range_checks.items():
    series = pd.to_numeric(df[column], errors="coerce")

    if minimum is not None:
        print(
            column,
            "below minimum:",
            (series < minimum).sum()
        )

    if maximum is not None:
        print(
            column,
            "above maximum:",
            (series > maximum).sum()
        )

SeniorCitizen below minimum: 0
SeniorCitizen above maximum: 0
tenure below minimum: 0
MonthlyCharges below minimum: 0


## Categorical domain validation

In [29]:
categorical_columns = [
    "gender",
    "Partner",
    "Dependents",
    "PhoneService",
    "MultipleLines",
    "InternetService",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
    "Contract",
    "PaperlessBilling",
    "PaymentMethod",
    "Churn",
]

In [30]:
for column in categorical_columns:
    print(f"\n=== {column} ===")
    print(df[column].value_counts(dropna=False))


=== gender ===
gender
Male      3555
Female    3488
Name: count, dtype: int64

=== Partner ===
Partner
No     3641
Yes    3402
Name: count, dtype: int64

=== Dependents ===
Dependents
No     4933
Yes    2110
Name: count, dtype: int64

=== PhoneService ===
PhoneService
Yes    6361
No      682
Name: count, dtype: int64

=== MultipleLines ===
MultipleLines
No                  3390
Yes                 2971
No phone service     682
Name: count, dtype: int64

=== InternetService ===
InternetService
Fiber optic    3096
DSL            2421
No             1526
Name: count, dtype: int64

=== OnlineSecurity ===
OnlineSecurity
No                     3498
Yes                    2019
No internet service    1526
Name: count, dtype: int64

=== OnlineBackup ===
OnlineBackup
No                     3088
Yes                    2429
No internet service    1526
Name: count, dtype: int64

=== DeviceProtection ===
DeviceProtection
No                     3095
Yes                    2422
No internet service   

## Target validation

In [31]:
print(df["Churn"].value_counts())
print(df["Churn"].value_counts(normalize=True))

Churn
No     5174
Yes    1869
Name: count, dtype: int64
Churn
No     0.73463
Yes    0.26537
Name: proportion, dtype: float64


In [32]:
valid_targets = {"Yes", "No"}

actual_targets = set(df["Churn"].dropna().unique())

unexpected_targets = actual_targets - valid_targets

print("Unexpected target values:", unexpected_targets)

assert not unexpected_targets

Unexpected target values: set()


## Customer identifier validation

In [33]:
customer_ids = df["customerID"]

print("Missing IDs:", customer_ids.isna().sum())
print("Unique IDs:", customer_ids.nunique())
print("Total rows:", len(df))

Missing IDs: 0
Unique IDs: 7043
Total rows: 7043


In [34]:
assert customer_ids.notna().all()
assert customer_ids.nunique() == len(df)

## Statistical profile

In [35]:
df.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
customerID,7043,7043,7590-VHVEG,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
gender,7043,2,Male,3555,NaN,NaN,NaN,NaN,NaN,NaN,NaN
SeniorCitizen,7043.0,NaN,NaN,NaN,0.162147,0.368612,0.0,0.0,0.0,0.0,1.0
Partner,7043,2,No,3641,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Dependents,7043,2,No,4933,NaN,NaN,NaN,NaN,NaN,NaN,NaN
tenure,7043.0,NaN,NaN,NaN,32.371149,24.559481,0.0,9.0,29.0,55.0,72.0
PhoneService,7043,2,Yes,6361,NaN,NaN,NaN,NaN,NaN,NaN,NaN
MultipleLines,7043,3,No,3390,NaN,NaN,NaN,NaN,NaN,NaN,NaN
InternetService,7043,3,Fiber optic,3096,NaN,NaN,NaN,NaN,NaN,NaN,NaN
OnlineSecurity,7043,3,No,3498,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [36]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
SeniorCitizen,7043.0,0.162147,0.368612,0.00,0.0,0.00,0.00,1.00
tenure,7043.0,32.371149,24.559481,0.00,9.0,29.00,55.00,72.00
MonthlyCharges,7043.0,64.761692,30.090047,18.25,35.5,70.35,89.85,118.75


In [37]:
for column in categorical_columns:
    print(f"\n{column}")
    print(df[column].value_counts(normalize=True))


gender
gender
Male      0.504756
Female    0.495244
Name: proportion, dtype: float64

Partner
Partner
No     0.516967
Yes    0.483033
Name: proportion, dtype: float64

Dependents
Dependents
No     0.700412
Yes    0.299588
Name: proportion, dtype: float64

PhoneService
PhoneService
Yes    0.903166
No     0.096834
Name: proportion, dtype: float64

MultipleLines
MultipleLines
No                  0.481329
Yes                 0.421837
No phone service    0.096834
Name: proportion, dtype: float64

InternetService
InternetService
Fiber optic    0.439585
DSL            0.343746
No             0.216669
Name: proportion, dtype: float64

OnlineSecurity
OnlineSecurity
No                     0.496663
Yes                    0.286668
No internet service    0.216669
Name: proportion, dtype: float64

OnlineBackup
OnlineBackup
No                     0.438450
Yes                    0.344881
No internet service    0.216669
Name: proportion, dtype: float64

DeviceProtection
DeviceProtection
No            

## Cardinality profile

In [38]:
cardinality = pd.DataFrame({
    "column": df.columns,
    "unique_values": [
        df[col].nunique(dropna=False)
        for col in df.columns
    ],
})

cardinality.sort_values(
    "unique_values",
    ascending=False
)

,column,unique_values
0,customerID,7043
19,TotalCharges,6531
18,MonthlyCharges,1585
5,tenure,73
17,PaymentMethod,4
11,DeviceProtection,3
15,Contract,3
14,StreamingMovies,3
13,StreamingTV,3
12,TechSupport,3


## Potential leakage inspection

In [39]:
df.groupby("Churn")["tenure"].describe()

,count,mean,std,min,25%,50%,75%,max
Churn,,,,,,,,
No,5174.0,37.569965,24.113777,0.0,15.0,38.0,61.0,72.0
Yes,1869.0,17.979133,19.531123,1.0,2.0,10.0,29.0,72.0


## Quality report

In [41]:
quality_report = {
    "dataset": "IBM Telco Customer Churn",
    "rows": len(df),
    "columns": len(df.columns),
    "duplicate_rows": int(duplicate_rows),
    "duplicate_customer_ids": int(duplicate_customer_ids),
    "missing_cells": int(df.isna().sum().sum()),
}

In [45]:
quality_report_path = REPORT_DIR / "quality_report.json"

with quality_report_path.open("w") as f:
    json.dump(quality_report, f, indent=2)

print(quality_report)

{'dataset': 'IBM Telco Customer Churn', 'rows': 7043, 'columns': 21, 'duplicate_rows': 0, 'duplicate_customer_ids': 0, 'missing_cells': 0}


## Quality gate

In [46]:
quality_gate = (
    not missing_columns
    and duplicate_customer_ids == 0
    and not unexpected_targets
    and customer_ids.notna().all()
    and customer_ids.nunique() == len(df)
)

print(
    "DATA QUALITY GATE:",
    "PASS" if quality_gate else "FAIL"
)

DATA QUALITY GATE: PASS


In [47]:
if not quality_gate:
    raise RuntimeError(
        "Data quality gate failed. "
        "Do not proceed to EDA."
    )

## Create the validated artifact

In [49]:
# 1. Update the filename extension from .parquet to .csv
VALIDATED_FILE = (
    PROJECT_ROOT 
    / "data" 
    / "raw" 
    / "telco" 
    / "telco_validated.csv"
)

# 2. Ensure the parent directory path exists before saving
VALIDATED_FILE.parent.mkdir(parents=True, exist_ok=True)

# 3. Save as CSV instead of Parquet
df.to_csv(
    VALIDATED_FILE,
    index=False
)
